# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq9PWtz27aW3/UruOx0SrYyI9lJmmqrzia1k3rWSbxJ2nv3uh4OJUISY77Kh2wlk/1b+31/2Z4HQIIv+XGzm2ktPoAD4OC8cQCapvk6KZJsZqzgx8pt4+AXI/LC0DOOjg3rtzLelV58dHxwaDwyDqMtvS/ETVFmHl6+CEXsi8wZjX7PE8MXRiaWZZYn+WxkGN8bHxI/yY0Q/o8SX+DvX6XwvdhYetnaw3ciNqCtV+e/Gx5ebMUnw0pFtgpC4/Xzv5++fms8AkiG8fzsw1t7bHh5YMSJsfF2hsihnIcQikwYZezREIwdgsmDdRnAC+FQN86goVRAr4xcGCl0EVuPZdvvnr82rKW33Ajsf+qtgxigwuVZEJc3thGKHYwxoV7gCLxsuQm23PXUy7wQxjUzJNAMOiLHahRe5kNXxLqE6jn35CQ0isxbeB/htYjSBKobWw9BPfLF9lG+iQzLD/Jlgo+gZ7YcAKOIep8UYi2MZRJjU0sv9gGP/6omwrB+hXI/UFUokwFmRExdV12lmjqKjLWIAZHLIInr2ajxlom8DAuaK2h+XeKgqCBinNHGXTVgNDAx+QZxF6rZqNH2scwLkSOmUlEEGRaJgjySBUVuBHFeeDH8JxJnZJrmKEAEFdhhoKkM+uqsSiA8KAoTtFyp9+tldRUmC3WNPQmD6jZI1NXHPInVdZKrq0yoq3xTFkFY3e2qInm5AOwvRV49KTaZ8PwgXlcPgqiC8ylIgYrFSN3HZZTusOdxOlplSWScn54Z8t1p5K3FmH/epvmICzhGjYFVsB6Nfjt5d2LModdO6hUbxw+y2IuEpe69RY6/lutiw65r2yOYSvfs7a//DrWqzjpnyfLKskd/nJ787T28sExoLS7MsWEuvOUV/oZiRfdZsN4Upj0ajb4x3gMhZwFQ2wy65SHR5MY6AD6CWTUSY+WFOQ4uMVJgcSCJoowWwA8iBLrJtoGfZCManwN87Z6+fv7qxD0//fvJGXbh6WN3Mpng/yN8+/70GMd5OHn8DJr2xcrIvZVwqVkriNb2jMgKaOQPL4QuAdPHSQZiK/jkEV1yB3HG8K1VeJEXJ2NkTPi7wqJFYjtIYwgnWEF1IJGcKXBJbcjZcOivbBD/ZV4AfADtluIky5LMMoGnpUzA0QMytB5sqX+AQax6PTY2MCwA7uTBJ6HavjZ+No4ODeQeutrX1lkFWaCMikAeAmuCcAO5Cq1aURAHUYLg0htbNktNfA/AfzH6J+ABDa4zEDyibiDybiwcng2NqBmswcKI5biXSbqzbP2FU2yAUGIvCC1L1RxXMGw1DWfP3/z6j7fvqwaxJspZnrkYiPjdqxdIsvDz3LT72463IissWYRKZAIESoyvJaEtQ+HFZeomZZGWRW5dCZHOn02AdoLYXWVCuOvFfDqpKfAFCFmPJJwmKQFNxjYQH1FoBqStKtRJXik8g+R8RYRFtqs77efQ5xxYX/iWjwRr+DhIlG8O/qlY/mMSxBaLB+ft7x/c49N3gIPvTdtGJKlSQQ6iwvLtGvE4EmyCZB0IkvzKLXPkriYs26GCoPq///5oUtW+3oB8wU4CERgWaB7Lz3HmEVlIx1TpZx1l2oTQ+KBtHzqXpNbEbryRPcoiUOrC8sdGsAbWFq5AcsznH7JSNCtowxQ3QQ5T5oP+Mx0Qv2arVfwHZTMRJVuhF2uUejhqxM1SpIVxQj+gTuvWUw+Uxmi0DOHXOInBxBD8EknOdYFpC9e1chGuxqjhV/M3SQz6IEzW8zQL4kIbCBZyUugglkNky275ohDLQuMtKggQoCj8bT4GFCzwBTbTfJNvvFQMv3Gj7Z6Xxd6KA69TDwY4/Ma9CmIkF8km1cuPyQKZ5LPpLQs0x8yZAVxqithdJqGHd1+aFVww/IAnr6HSS9BVdVvfgNpGWwzYVvgBcqiFF2CeJNsgtMlyAukHuh3U2RLM4UJycBO+AJbG/pB6gfafOdgfaSvD/dETerBgQw0ePJ04nT4CEBdpWdPy/Swuu+ygNaMRcEOK6F1zytT3CmFheaAKz7eSFPi22aqtSYhhYq4IutEGm4KAgC9tUvP8XQfn9Eqiwk2TJIQSy5XzgWyUc7g/uQFHAhwTCzSLe51kVwK4/xDUGFPGhSmfmZd1lwEdkXcFU5flHYSRbHCTKylBauYTcQ4mJSFC4zKQyu8DtF2WiR+sEzR9gc5KtDCkBa35NDuv8mfG8DJKQ6QFeFOB84osWMBwwD4qBfoORFMxOgFge6NTBNoDpMAy8MgwzkUOKK90gxKZPFuuHyxBXNRvnFwUMBYPtI9lSk4EVYgMZe8thTx5S8GaB1G5mkPFkBuhwD5uHKqq+BKqE40MlQOcUgv346/BVhXNA8z7cNoQPCJ/7N9gi0ACadLBNqgv31kLeK9zgwnqO6eSLb150Sx2+c+wzYiF38E/8Y/pvmImmEkULe61QL8hlwoNHL4knwMCBJjWcfFICyqYNIOdx9EWEN1gxjNQtUUVTKhccHTYya8nfvLFRy8Lq6AC+KQNDiKfalOuQfuuV95SuJtyoTysHBxpl+YQuCurq4Cdg1rm4nJYwqJ1lqF11oRh2Q4NvGuCwJRn9M4NyKobKFbBFlsq5SCboFgYKFv39wfocF5k1sohP5AkO4FacXNbep5fDsp69FJFjx4BO8Iy35A0TEvwAkIhSFxV0ZPfXs5gSlsWmrSxJ6OaMYBWMt+ijjWbiQHZk8aT6wC0IekqLI4+6QKaWJSrlQDLaD2f2NjbVRcnbKSiuO/H1wKaYvVkPX1s/PwzuJt2b0HpHi6G0S5HGPcWiHE+0EZe2DUG0ODBYIGDfzSjjUbbz9HTpzRScdPsR5EUHmrPvIwscQMKMLUyMJtphu2uNWitzHPmGBmxkkE4YJaZ8ZmBSZt25kxXX4xXL/C12R3ayvysjcA4MIqZM1l9yTUdIfFCUDWVy8onDVKhSYgxeutjAyosEo0omGl3Rz64oGxJoi8qubYSGcfBh5dhcv3aK0AwxOtzAB0GmjmJbQGObivvYGuojwpwhGPwvVp0LHu5SkKQwHPqb5kLF4MTIMXyroMCpEMDak7ZN8ZLcAQ2fxwDyn2BFsYqWFL4kyOv/zWd3JALmXlpICOPg+JHDc8B/x90g7tC0Fs/sqKl64XrZG5Gy5aHI9n9wy5lL78LsLeVwZa6bHMHgdIVLEaFFrSMwPFKkziAhnpEyh3h3xu2JFgcZU2vZC0zoV6D8Q9WE82yvCMDSX9AE64/+AvNxmJHPl1Tq/2KDKibko7xAWsb1hO0sOH5D1W/bQzAgEGKMSwKtULXwOHTSItirmAbGbkXUlC2Cj0bvwKKMIDMYRuj8KJFAAUoGK9sVwBegZIRbA6qQW8m3B3lC8le/ErBLb+hZFUgNAEVXUs7vHMwuAi2S+4sS98DQVVEZegAwSfXbrE6OgT+RKTtqRTH9y2+ANtiA27bVbt0Q6ywNyx7/gIgrLMEEPeO4gSZzsz6fPZbaI0S88rsgn67Seyu0xJsNpg8VCg1tdyuF6BtzXHvbbnh2HdG0YRVN00hnIYLPwycOIloFqjK0NeGJNUy+ZHU8mzHcVpyp1eqtMIH7OJoGqLfbBxUxUP/zI2s7h/4QXGwPTyYHBACQAB0g0oPkC+MgtsEVw+JNOiyTWPN2XnQ3PCsgGPK3NozL43Azx0noA+hXQ9ODSfa0kiGyY6iQw8YHQCWtLf1soCirnnh5Q8gPhmfutPwwT+5PwX2EmG0/bpUWCNkHyW2psVqkBxIp4GJsbuz9sA5a87WPtq8fdr+byetb8q6dP5PzFZjnsboWpU5Ktk8AQU8pcXRgdnjwGk1JXzbOx8NbVeImz4T+hyrdwxn6nFJkVbrQsMFNaZL0cuWWhGAH3CbbbBg+uuZlx0vOkd/nFbt8j14W/URFfXesD5/R61/h52pLgkid+g7tmi++9JDaXvt3sEXnTB2P0L3exYdousXuQ3Xo99Xvd2016jI5KmTGJrt7RKtDH5VDPQO0h7tB69WA6C//U49uM1XX8sduYsy37+MoZtuFX+i7dfl0XoNIBfARBTWBNcy+MROMhqR8FAPd3cauDCT1Qq9FfNydktHlQ9Hhr67TEtXVm05c8g1t9g9Mhno9HkTX4a1BAmW4TLVDvwUdDDqvBDbbIgzxEifpdwNtKW01tuQ/+O2utIf2L3ht1SfBpKg1RPdHB1wgZ0isUxAmWn3r3Q0DDp6c+1lUZlavdGY13IRIQRVyKGYW8MqqrYbCXS+K0dVNtNdyvg99tCB83xMuYkDsNHBq+N8pRktWAQhu6FXIothynGmfv39+LmxAwII1vVoKPgarpMsKCIOvy7L4zdvOC8LYKQZ9DpTuT8epjZ5GLfFxK4wWd7NW5TBmoZj0DBKdNKhwlLpKceKXUHyMoPc9bYeDA8o0moRAzv7t7pe3YBvX/LOceZdN9k+AnicuxCLa5l0MAZfeno4NuCPDdeTsSH/s+3O+ngF1sE/VhC1ijgiDIM0F5Y1/fHp2HgMYI6O4OJwMrEp/hfOLbgeG9Mf8c8T+HP45Em7IaBaYM54HSKcn6D69CcENEWIP9aAnsJDzIKYPu0Fgy4nY/0VJZNhvNIX22Ap5ibOg2mDvx+DgQnCDUR/K9BKQU+uHsQrkaFeIMlkQfv83CuLZOnluNCC4MaGX4CKm/NLFFzF9KndHxizKH9lHkRjTMFy6xbyQqT5HGYjWWKygZsJsLlK1A/z6eGzAVsRQSzBnrnK50pi8a15OZaZdNCp+RqgUgaJS/3ExZ9I5Ju2cVsTar6Ll5sMROcnYdmjAcMHQzhgKoPSRFa+i6i4V2TfUMAjTAxMjCQKisBX5mcjHubSala+VgHcJCnm5qNlEheo0T/gy/evzKYUkk8N64/n7z8Yjwy6N56f2jMZtwJxQokxqwBEIsoKeJLDQ7aCbxccIA7WovCKIpPdaq65DeteGfZrxpPabI/LNjBMlXCU73jVsEc77lTaTZxjvtGE8WPfpWBjKRJrwSDyJYyiyE3b7oohOQ1OKq2svLpSM6RQJbHfa+ETqAV4ZJkeCIP7d69fvGqyajsxQfaxtuvU4l9fjIG65GYyRCXhdwzDBvxrTOmCUgdT5zFggHUvSxSx9UJrqBXSw40R395ORbbUTGe6lOhpCp0BXlLETmrdsCJMskYCryi9on7TvpUW78rFauQdG7Tbr163szbdh5fxOI+jEgZSQTekAWVPknRFwMIHwVgCO4PMbeTlMQeLiNNk93O2fMhStBnFJfHuysxcVRKmOfUymTLaXF1TLTpvvEj4HzgLO9u9hEdWXq5Wwc3cdNJ4jUpGhKIQc3b2aZmxxe6UyulthbXaOJiKa7eSZCiRR74blixgflE+l9Zludi5WLtkRc/j1PGyzNtZF1PMclB/LkFBIju5sSjmLRbTkv2CGKzqVqN1Ihwtw44eqMrBl7Dwwm6tYfar81YXlJVHnZZ6GpDRUqR9WhubxZxTSWWHYJDsDe4oAnRzVHFzYl5LPSTFoa5/MZ5wrOBHB6wymF1M58kvJpedVU0mRucD/1oJlHHARAF9b8F0URNHaObh3OWoGoN1mWBwB+cxuZhe2noeUq/XB/wDlzrXrG4N9rTN2l7ztRkMcvxgRfgtAvIKM0oxEZmDA5N3irNew6N39GQ04FrKDBp9MDjRKuN5uAY3VFWQ5hU/bYWLtMqye3OtZ5ZMutFNuja4e8UC9bHMaUq6vcHZa4/zoYkVS1yg4+R5zl1HjSFTnbrmGDuCLbeva79IyZjT9oaeN02Ri/+26MNwBWcbZJj/hk0l2c4asv95bY/KuLhYt/Qwh9lWOQW0Ij7knjHfTYasX9xjUpIT+3kbOWUu/EamwiN82kpgmHACw//8d18CAycxoE39eS0hSCL5bpt5kbtefHeJ1c17JUV+jawqtb9CT6xKdeXKkttdrDnfoCEfMOF9j2BoGgdYvrXtIS8yu6PlKl+WMnFwC0Yrwb61QWPUU/dtmmOK9gqsBS8G0yNvFd14QA1huvFkvj6l98/nMqWfwjSVGsSKF47jgKt66URgv9nGz6AFJvrAKiRx1aSoW+gbX71+arV3C+im9/79BDz54FMDQwH/Ai0mi4+CtqVhrDNDnuaNW6XnZxgUw2TSyMvWohbZiIDekVYldqDzbnIuFifxJ5EllgeqazppxNUw6+gmb83mDZgON9OxsYPf3RRg3OSMwTFdgVKFq131bCef9WCMkJAlqWUhzB3BNX4wpgQXfnWboGfrC7E3PpS6nPePfA92zfSZFuz6qz+OgrblYAwl/wuUA9gGTNGWlYNzfA1CCAQDVoW7Dd/ZndwPqApaAxWwNZ0cPgYzC/52tqGQIiaEHz5msZX/pRnFaRgUbr4RolBqHK/Bvcswes/7rca4yWqMCQtjuc2q4S0fB1twg2kv0Sb5SC4Dr5bxvsNYYKZfjiYjpXBhRsgqif2Wm7zilvcKBC1jO1svvCYrDHFua9MGtdHZRiVHjRKFkaTJEHoHBtYQnzWYAPtVcQGhvSq3pNxxz/HinR5XysUaKAR6AkjEHnhZQXmcculurKkZjCsHY4OyLAUYm2h54m6P0O4EALaVKGHIPdHlVoucBxS0guoSVQxuP6hO6rw+QsdLAZc+EDg2B+3YDf7vg9xXDeUEjler/TdCKnl3YKk20Ip4zHnBjgIh+AgJDcoBY6GpDDP0N2DkiTN5fNmWR1ichPWMRCUQi7EpQV7mM0wuwr2EYCaDjZ538kKxNjGQQ/xlmeN2SET1jrySAHrwNzAFYntMIgYeoEyqnnJKbEApsV4Meiu2686iw0agEidH78Ti0gmWbvehrpWURXMHBNZBUCB0WOzaCOATwKMGxtTjtldUFhf49rKygVH5IwdowlbKcHqKwoq8ic6ytGRwgPhVzBKKXdQeC/qrgQq1oacwxm2ALqZW53fN4LSOhfT1xEuoJ5OVcCB445dLuLmjof4SHS6RSRA1ErBnyESN15Zt4fNOqd7udAoP2tY7LAfWbONlhIRUvXLQTXkvipZGjYApfd8lT7JZltp2UEaCD54zph1GcgcEvgvitStNH78EIkUL3FXVrT1VcDZdsCZccDgCXGh2BczZnirLEDSBu0mAWVGB0wU6SIeTyWRPO26SYXyZCcVdJhvyPMO2T4G+R4SJfLT1mzFj9/otraiQNp1tHz0iPIobGCK8IksnIlxqD2Qkad7ipXtG0qs5HIi0yZ71xNl6Uw32DFUNFzAMVuTQTDNkxgZPtbMKblzOh8hb9D1YIQzVdHfK9+LncP/aMUW8+OQBXmtg5/awocQqmULKEpWAxgHGL/X7WR8FaIJEsnEtpUDXz2uJ1ZaahOhW0roUdrhdKa9CnLx81A51ylR23HQODm/TReNoMLo2MnDK3s3DFi966UWOXl9DlSFa6vuF3PJ/ORSn7QD8RoXTq+ziBIxBg1mZpvCRwNiaN6O9chhw93BpuUrSGQ1smuAp6Zo4906nUv1bYdjh1oi2svZrBKGesu4xs3Y7CE4A7k0gHQ++LxgTlWERoGlxNQOjEa2KK2mwUktOUIgIOJ5NVI1SaktE5fuxGUXwkHmmtR0NFMhpXnvSKeDBZbWpQeZ47U/GuLSNP9tWtOqLzBFrdYF2L2l0q4Pc03ij5eKynU0Qi5vCSuscEm6rnQtS0wfu6yB0cYLBrUknmsjKKZG0r16xr9qKarY0ChIQ6kHwxSnSjvego57ZbN6rJ8bPc/CJGCdPHpZZkxQbkbU63MRpq/PdsfTm2ki4rXybxlNEVX+uDRXrybeptqLUy4FDC+siSosdb8uz2lEcZivopJxtwl9TQP7/rIv8E2kOUtI3Mh32LJpIChq0qbu5EFiLn9rjvekPowctvtjN1Z7hjvWmU/Ssy7BC4Rn9Km6PjLtX0l4uBjQcHynw0bR86IpNryYMVhVM41/mbQsBlxyUgdBjinbXlKr+6YxAfm6T6i8xbZcfXV2yvuFDVgYOCCLFQ5oItzJf2Xu2y35j/MG535lg6sRra5XRUVQ/4KleXghs6zi2wTllYNDirlcjTAzPBytRtL3lyk/2KJJIWKw4IVdxB7wmjcdcjrdAPA+279UZaBi+bSSzo71P/QTyHnMWNw+ux9S/tfONHtbePG8W+DrEncgNhkjZV2LXNGCStNBdeRQ08lwtlPtTTaBWNEKnSjXxhmpUp45uVG1vVuxGnV1x5QC5kLy0hwttQScsdgU6QBezmfHTjxoGN41TMPwySnMLhzims3ZcGL7cwNnTkMT+xtmIGz8Av7jABqaHl7UZmJWxRB8HeA3kFnWUCzCMvCSekdcUBeXrbkTV3Rsubq8CjbuWg2yEpbe8IWdHXgMRq36AqppPD48eN6EoIT2nsxWg3bm5KXE9MvDxsIMNpfrMp84zTHEC0STmEyjYnHwomszNXz2wYbzMHPMDVDCqaV5hV10CJBe4HNgzJinK5N7KTISB2GKLz3RTmhz4dYYefOhFC98zUEjPDImK2MUoY+CFPUinnknHn4oCP68V3nwQvaHqFntwc7WxwWyfz6NOFenPcqZE4V7R0si/Pq9kw6htD1aaHFNezy9Mum4chUJTLkvKGy7KN3pZdq65KF9zSbrWC/ICNxbDKy7UykmvJlcWrO+5eHWvV5IUoMYl7+TI+K7RjUkrU7e29eXMW0CFh6Cd/qMMCtooQisjpt5NkEZ8Wg8pvJkWaZW8WzGXrc7F0wsRH+plunYL60i9EnFwEzCzsl6InuilvvSu5vSLS90zZOtcX4mSqzG1aLGV49hruesiu99waaATLQWQ86M2hMqoGDB+OkfOvaQkUHWCJObMkf7Upg9lNp4gFiwlG8wbnvZcOdws6+gvCzv4X8q6yj7vtzgrwVd76yj94P9K6vFPJfj4ZzyAH5SB9FeXf9VVLQTlbz+YWjYukiS0qlt7XIlGfsE3di0j2eSWd/YgdBChCvIiscdKINIzvrZrQUhP5Y2tJCIHuyr4V0qEoWWh2xQ6lVxV1MrniJB/2Di5rn4JRHZhwiOQB0O22Tt1DB8GqagSWl5XvcmMDcC1HGkfAmi3Vnj2HhhEkikvshVJJ/Pb//w2+tZ3v/3t29ffvndN3BB31X9kFRl2rXOqWsfi0fGJjdbxSFA8aCkvSj9InE0RhaZNyedaZVRj2G2p0VC0WlI58uqUP2uFP/CYoPZAz/n0IJfQPx4aBRYZHAath/VFrmb3tAm3nHbZTOGlllcm2eHu5/gLZXHamn44UeQotMPbvjHeYRiaRE1oTCcT41s+OQE9DjyZC8/xxZQsNCW8XKhjivHg0yxIMu0EAqSNkwt5QhX6UNbJRXVC1aV05pA1ZaYTEsNJdaQT1Xhc48pVW7JxHRVeTZyfJniEcN3AI2xTq8AKWlWsatzaC4bTOJ6nCJZXFjgwC5SXS1x2yik3KElbc2Wa5vOtF38iib3A5XojL2FuInLofEx6ISFOBzQz+rK118hSIORNB7bf1Ef5kP0EHcCssRwX0LpkcSNjZTDwJ2PDau2OmOJIKWuURjTtcSMqC8LD2VtALQ9XjG9o/D0Z/Niday8ooMUndhODKXhKoovCVdyWXACjcTzvyRaQ1sIAJRLXRfh4IKvAvKFijnM1NuA6nw9MGXC4BwZF3HN+JnAPZQFYd1iFklJzFbcK9yYYV/jJO4udsl3iWg1r1bmOF9XBcZcYDp7q2kKdaAzCfnmFqYWtHDzNBDwCwXgSY4aIN1OnjAPPoiOCqxeJPObEbIXoVAs9CqbduQO9c61S6iS81hAGkRtv5SlODdFHIlOTl1s0lPI+D5hMoiEOUpFCCoz5MrJNBAp4AlaRMmOsSZe7rrubHA9FM7s6gebo2ET+MdvHMpjkB23rMIw5sOO3+4/A/c9/8+kGZmsdAMDctbus92b6mQH7Vn0ecsxDy4ZsrQT2zFwl6VsCKx+WT9WEmWdBlAaEfswxzHgmvHX5UWRJbvat5IEc5IOmqrMqwVIio5vDvrQga2FkCBOQpYknd++ePrf3LjaqrBAmtqZRUSc2rRKX2pu313PvviN/b1qJturbW3mDi2nUbRkd1JeUf54bTyf4j2nrthVkLmsPtOOIG+xR01IhS8+kOSA8OOtwYdpf7RgtnlxCsL6s3kcKhzYulGY0PNxtl5a4zg4TneP+YA8/m+BD/dyw6OQnFfXEry+AaQS0k9i30igZIhOSMiYhMsDvKSA9fa4oYTb+As1+5nV8uKbGzd697t0MgJ7V/69CJXdMIRigXOpj3+yznWp62TKAeRqa+0GLv696y+BvGP4WE5rJZfTJB8ZOPWM6q0+e2+HRs2Dd3l+51FblwAEa/NKHXlxJO5X1T4NIpmO2YkkPVSbrPYWweR7EhdJHepz+9Pl9T4BpKAu17tNe8elj3N5jEwbwMJQBwaqBy9yqGBTcfRTH5o5cEQo+gYv8UOIbgrSfDtWENilR0eHhjCaJ9rvdXQ0M7l2581Lw3fZ2NPZ43M327TMFGwaj5smhk/ek6dz9IB9J62DU5TSOlBAZNZtv+Hsa6PphDRyejTRfWH1HZlUKOlE3VJ+YwYgBWdAgI3jBtvkVGTrWw9EgHdMho5hLl+TVx31m4AeCVUE06lXn+I7RziCLnHdn4Wc8ON5UAywWQ0fjxWk7UnELlWqdPK8PUuS1Oj7bEA8/DNC/55mbUZoy98iIA9lLOiVjt9RTlYnQaT3V5DHiGinVY8dbguDcFVXkcvCcej3w1iqkrDJZSMXh6gh3juFIDnHj8dt0NjY4hHBtYqd8sMzwmjtXbWqn/tcRZ9GB8qXp5ApU9pZ0j5tEyHWBnokG0e3WnoyNG3s0YDSgxANrH3z4H0HW0T364I3a5I03erId7sm205Ptnp6od4gsKp+vdQY546AQcAHFuAQqTJCTgUcBoXyGFCSjcfwlHMNbZEFWE5YGS8TqewLKLQ09xRxohDtfJ/5XHf1sNayGPRwC+C9KcA26m+Db4TUO6VULv+3YRrbs583VbR/xgJo98R0dDVCiPc5q26h+SLqTlwuwgq2uny5L4faAdMzr6lqAX0X2ZRhcC9xX4frW8qSKx1dxd5wYJQaIU5oVlHVhSf1Y2dRSQ+rLlSzrq1t0IfTtGqvOZznwfC/J3BVjtzMx8PteUiS9eUvLLB6dnE6fVFuj3QPiXcTaWbJGGuJu1J3RslW+AY8Tq6Afafzj9Jzo2q/oWqPkqrd3m6j/q8kanDDG2ZjlSduDXfD3VFbC4Y/96McEuYuwa6gtdGVfx3k1nawHf5VOXoTD+TNV6oJ1MjZ6vuEBQ7g27ftvWVWOeXMDS9DRrey7AuoxvSxpf3+ky8hBO2AsG2l9iASK2Y1+qFQN8gTJR5zjbuIwt2z+XkLlO6K8o0w+kkxzGeBSa2KeG3hzNpLG1YnILhlZczog1ypUKGU6EESqakmIVT2CCbXqEnI2qxKL8C5gcdOyqtKknwnD767N9UKU39ibc04Lf0Zxfn5h8hWdUJSW+IByQGt81zSFSJVkFaRMSshxGMbHk5I4rcH18mUQqDwJudd+jrv8tBVa3KzVpRuJH4xUIGSE71xnAcwzspbdt5L/0zMM40apB8KR3TqQLxrJfSISLYvqc04tSwl3W1Hjn9Ie+HSShnmGp7Q0Dq6v1pf9IJtz9SCdf0rHRJxzxtOV2M2vlKzhbxPOV1v1gOyIOZsTLWMFnIYg37gsey2oad8hp38rhU9ZdKVPza/b/QzLbbbZVmO8W1hzH6Nv7QEfbdsHdGsPfG4hXjW+m6KcBP64ihILwZalQPPtHSGCoKAaF5c29L7ArZCDcQHPJRMHfkCNVq1qEBAV8FZa0NhWp8hgty7MSggwLZBuUK2odwMx8voEF0Rmp0iDqb8GT2vMcUHSlzsbr7onXFff7LiDCAD53Xe0TmUrbKSAMP+M/4xxWWHbkBR3YPYHbSfh+egJnA46/MpxINfqsn0QZMuElR+4Y2u47xt3ukpd7VkNX21p4yA26oJ6DDEzyWrIF80W6ISl7tqbBn5UwgQMMRP50LGUrUVDLEnP/g3RFCwjUWwSv95fpc+cdopJhNIdRDwKd7gxm0t18hurzj+ClM5vghJE4OP6zem5e3zy8uz5h5Nj/jgXiv0QbL1wPiXC+9RNoSJnSX5+ESgYz/DCNBSyubvzjkCxk1qL7z+8fXdyTFPn4OcIsLOWZTpR+hjdLnm8lKncMDZu8aONf/af6ovxgr4RdUp/kszS5bsVnuVYY4AT6fGukcaRCTDwlxKRyEMyx/i3oKDzPkM8bgYDHypOdPCL4W3p4CRQ1Q5gFM8ExVQJGfqh+q9Pz04AJW9O8Fuvn+tTDqRzNDMu6iA1LUeaEvxMnoqlorlcqN/6afpPDh6zWYMhy4igkHuuyg1Bgs7f7KqvSxG4Zzo4YLRlGRI8LjEIiKJfmTqmASEdNcdHBfjw10EokZfyJ6mlm4Jg8LzPGox6MYgaDir4as3b9fAYWJ8w9QQ7dMIFcESvzl4Y6vWd4V2LhaRmB48INREIHh2bJfzNEHh/d2BgrDirxQ0Ba3Xu5Yu/3wFOGohPHmH7RySDc7rlD1SXgPC7QFAT/yNO/GsMUFfkcOdxhAl+Nc959hRA/MHbGETOaMEltbO3x/adgUkZIc9+M5/LLw7XrGja2kKJdGN7OOsPfNHgK5EvwfHv70gvBR9qcCpeYhoegiIjpkRWGNkiRsBxvFMBX+wJdZqOAa3K7Ye3DoC+jp5OaKI64Kq3A0C4xwqjf3AgRLGiROaXOr1fGi7qHKaHhiG0sEOPaaVy2p+OOWYzx217JkeCVF57lcpOieTt7wotIzw9+EKmJb44O3lzfILfMTwgVXNwsPKWRZLtDijrp0zpIcWpJ3R53l0j6ws4qsANsmq6I0Pu4IAbCHB2CT1wAxjiaFYHKryEyqbEH9wxCk06L8biGwbLOJUv+MbuA1eFbhj36gmlm3Pl6p4Bq+RyfinvekFXc4ZDnKr0E34kU1AYfQe1iKbeDqXcQkkZs68htnd31QXICKvb6QFGxGJKoqGO1J/VlOA59/Nf5pXerU7VrpYcVAv1kgQdXkGFaiVO3olsqPWqXtSozQo6l7b5EcU04zP85VkLzjk5C0C5OAAfqGWuvzw9P6HnYJHqz99/OH77+4fbVnQxZiM3bSzKFZ1PMdU+0YkpVvPWYUDYYbQA08zh3nSyU83ndICOscq8SFCAlerIupxQx0af+RJP7OuxGiOynsEvRQ/WyhoQDetP/wcbZg6h9freURUPlkKBvwPCib7zSvz3+411LfxevEW5ks4RmNi0Bc3B70al1tTGbMnHdJh4j5Tu8UnpiMu45fLwocTyvCFCTiYP1OmuAEiBxt+kGTYxdHQP2w6qVN83wGVDVjMbol632Xm3LN20IQ1896FDDBco8ez23HV7KI+ksq7EbqyST5vHU0X5ULAFnBfjlzlTNjYEMBQuhj/xIfmAjkga3oKrEU47IxZ4hfJgNbZP1sUNZkSD426y7iBiaASLU/ysKUpS3G6HsmkykNbP0C4OjiaTyay7KZJfj0Yj9+TNq9M3J2pCRiPU3CCvXEFfXbfan1Rfh8nCCw1ZbSS7paB0dqzU4E9qeFU8QnZGAftfI1Fwcw==", "blender_rig.py": "...", "app.py": "eNrVfWtvG8mW2Hf9ikobd9nUNFskJdkSbXqh8XhsL8Z3FNv33iAcobfYXSTbanb39EMPEwLyJQsESHCBfFlgkSB7gSBBsLjABvshQfbb+J/MH0h+Qs45VdUvNilqZm6C9WCkflSfqjp13nXqyDCMN2Emkhn/xK7FlJmvEu75UZd5ImAizXK4GcFNmCURPQujTEyj6JJFzI2WEYv53A/5XpxPA9/l8CzE/7OEpyLkjDO4uhIpfMrykGV5CCDg+kUQ5d4s4ImwDcPY85dxlGRsHkRTff0xjUJ9HaX6KuGhFy2LO6GvUuEmIiuapfk0TiJXpMWTbJEImFg4Lx74S7FXdEyTZjyFq729WRItmc3UO5jNzJ+rhyKE2Qr9ai4yRz7Z2zv79Zu379mYTYwXfAkYSQyLGS+iJBF09Z4Hucf1ZSavvuR+IK/eiThKI7x6y8MoxYvXfIovL/bO3718//IDwl7tMfj3CNZgOU3EiJkxT6PUYpGbJUJYzOWAdotl4ibLE9612aDPqAks1lIAXICxjGAVI/YhT6YRgTPo0gBoxxY7PDmy2OM+/LPYsH900rVkm69ExoOAe9TuxGLHg6HFBsfrDd/x2JethgBuePzYYkfrrc4AA9jmqK+6PFlv85bf+EtqddxvdHjUP30Mze72Xv6Ls7fn37x873z15h3MzThA0gNaPXidh7c5Dw+/6g0PeJoCaRyIG76MA+EA0LlIjb29PU/MmDPzA5GaXndEvUaJJxJaRYknh4fQ3ovseTDFRVFPgVMaTxJ/bs+mN+UTexqI0Ku0kKvifyqgUYfFP90MeI5nvhvZ0fRj5evi8TILKo+BEBOR+lF46NkpvmkAjX3xiad6rLEAWtA3PHF9WFJ9CxNw/HAW2ch4QHX4+QwJZgKfxDxb2B8jPzQ9i826bBYlbMb8UOHLnzHdSNz4aZaa6990C5BfAMxZCSIFThKeibxv44+1b419o9vtVicG/c1QEOHnABBkAt3NgD299NrPFqZp2EE0NyzdLQiHPMHGetnTfLnkya2JU1Zrz2GyhAFga9PgH/M0E8iJqzvZeQB8TggphmLOgEoBhyjR4OebsxHb318hjEmHeNFBVu9cjKw7yZv7++zHv/r3MBovd30PxaOBkyl7Lb8yukwEqWAGXHxR9DgzVAfUXHbSsVjnzzvdaic//A8tBNiKy6Zw3+nesfgGXxoVgCL9PheByCJWAQzUgK3NyqNFjtQDnfXhhbyxGFEUA2FYA1n5KlYfdXoAr4t9V16mYp6DPE+dLALxouZxx9IKGc+Mr6NkyUds1VnmQeZfAXnxTg1p1RcKaZ0B0FcWde5qwzLbup4heOqVRvdBoQ3XsjalTuo3ulUYdvyy2xC7bO1GNS47qgJ/B3QrklFtAeiRRlwNIT/+/r+zD74AFZSOGA2fbZ5YidPaZJhJUr/bnBI+LGYDc6kS0pZZFb00Z/ZlQPNo/XgaFHNs/3hldq58TwCBddgXLM0SswWKbNHpApdAs5SZAkSKfMniAJQpYRvmVJvo2ufFlI0arxFh7kKy+/tqjaS8qTE16RAXhDSIEyVrCnli8zgGNJjGWdkIKA8mg0JZykASKDUoF0oe1roJIs8hAbC5k2++/QqezEUSrXVSUxyA+0sk5KtCqHQNktiXFrtCqSuHVPZ4YfuZWKZmt21gSw5GiJOIwBdXYvPg3skGI4btUaSGSMGgnZmLthuan6kUtIrsjJa+rkQCalIgaQZ8Y18gVT7/MUV7FQBiy1Qx36RTAyBF99Xn/yafUevP/yuEDw9cGOTBkodzMBqNutYttG/gTwVYllEqFQQISl8kIfy+ZdOEf0LJCdhjIH9hrkC1YN6m81yEbROTEve+ab2mVnpWxaQqH3cuYGERh5V5YNdTDsziCmnBA1CwDni3bSAk0cFUQDsLLdZBlz1ng40jOi80hMsD+AlqWaIE+BR8AQVHD7QGnIYKdvzNLVt+/gOQgUjw08qoyuVO/dDBj4HMEKfQgaHpdPvQRmAk99mvlApfkjpPfVgFxtEB0rpxwxJrpVKhm/WR4Dz0a1Y81dNAtrrWY1UCA5QZOQGTi81L/ePf/O3//p+/Z6vrOwVI2TjGd+F3oWJq+qQLNs+7N6/IgTDO8gytc4PntHhAMEDD9GRBVyAOyXFJfE4sBi/cnHtJHguPPngPIy3Qgq9DcKpyAHGnTKsrX1yLxFlky8D0xr8GeWWxhfDni2z8GI13fK8eX/uJGH/NQfJaDByMmKvny3ROV2ry4CG+lJ4oO/yKmejKCTBTu0hC4FT6s4QvBdBwAsK8IiVAMsUAlbxWMJLBSEDTG91NRT1oM3pgwtLFPQasoXxhG+cF9mi5LArtsCLPPP8K1NRtIMYdOePRSv6+i2+een4K+uh2NAvEzVMOUjjskcwcuQId8KfrFGagCerPbnvKq9EtEULPA9y5GaiDkRsF+TJ8OufxaNCHjqbcvZwnEaip0aPBcHA4OGmDPSXTvYeOb56OBkfwIQCKktGjU37qcf50Bp32ZuDLBrej9BZs4WUv9zvPW0BV501fpf4nMTp6HN90nv+f//Rf/92zA2jwHFs9R51jwvIi1o33+RSjApxsNZCJcR6knLV1MNXg1RDFiTgVovP8lQBBxRMkCxRbb84A/EtNnM8Ops/ReIbv5QDop+Y6dOfzJAARDQudpIVP/30eZYKafA8MMzOyMTB4ZmK4wMYfZpfto8gAK7ggJCTpkiC+Rw9nZvzZ1XiFL8pmSO6NZsafEQ8MijbEBmuNJHMg8mh4Jt3LiaSJS+M8kDEMB/zvA/Rpx6smCXeqJAxGy5+vvldjq5CwZqbEHXdW8POuo1F/7XvZAiis/6un68QtqWnUb1BaG5Wtk2cHRG0QXQPx5EGQgrIX4VN2k/RSGL7Pg16WwCcgZToN4PKz8iNYYTn8BpHWORONhx7x3yjB4T8FP3Duh70sikdHSLLPOAMJMyvmn8F7kY07YLDy8LLTIFCjTpkn05PpbPa0ZIPBIcI8myZ+gvIq5iFFUjB4Fgconn78q79+dsBL6tzbe8R6P+MfmyumQJ1LhJICxLd+uoyktwC8hhYHjAF0CYtmPmh6CsoVQRNwK/JQmpqgC3GcqK5d7omQewTOlIqy97xwD+CyUAtdm72AhsTbnriCh1dAUnkphYHZ4WsVOnwENgFbgoUC64xKF0QcBdAAHpcTsBlqHXgdpksfox2IySv/KmKhD82RcSmuAvDtvb/49ktSdVofOdfcz8w0s1gIpAEuawIWiEjTqnKJBXZNaAFlAR1N+ceIBYEgOwDHSuNgf4kg/pItIxwrehjQml+h3VTolesFMB91pSMUaTYx8B6s5CJUgc88wK9xUfK6HpgJszf79umpRc1mYO1bqNXcMZjl8KSzRIMCPemqUOph407WB5PV7s/Q4StDJtQuDYSIAe5xYT0hdDCpqmNIuA+icJ7YL5MkStBORh4DOxKBQ1vomUhUxlDiyEQyLjGZLqJrR8/DiTC4A2ub+hhJ5mgD0eIBqckoMy4895agBku1LKWwH8KauJmMyV1iaK8JWpl31MwGgy1Ee0WYMPQv8yyLQtsNfPeyi8IdcA80lVZF3Wod3ohNcDKAWgxnXUqHsKQiXC1p1KjpPmK/UfNDa410EJEkkcuV+DRC4nBFkAc8UeY90JeH1mXIlmIJNMurQ6raTWgAGS+J5BQ3eMAEHyKkuDT7/Aek/x/+7hCJQFtGKlD5wz+OAKspm4JKRfbdYLcq2XUmY4DAjudR8PmP8whjxIoVw8IeLldcGUcU0JNYiGFt1oN91XnFLeHBuEucoJ+CeEVZiY+fs2Mwxwn3aAGqjsH6iB0aiwmKOwReRi1isUDM4JrEODDLQgj9yyG9A68Az1fCmc4tglHGyptomXEwzil0Di2FgFkggzvycp77HjI5daWtWbiJMp4JtGg9Ea1BpKfgBADUlCLPCB4QkvFpIHSQHkFKtxdIsLgGC5hfChWbWAPsUSxejD8kObaEhU7GRiG7mZl8/gPG4LtGKevGwBbnWrx0K5KPSGwAJMPDT7wq+25LwV24RUB04Augt0bLKKq2NC0L81NaNVpcWgf9ZKOEkUYgSH25PxEoYxBExCxBa5eZEamRRfSRnHUK86Vdu/QBi3Uq+8A7oEu5WWTjLzThcH9hf/9wAKJyIL8W4Ryalbs4Zrd1OiVg+XiMH9pgzfuZQ7M010mvKz05+kCN9aOPg1I7VXYWXYrQWYgb87Ey5DJy0GYgiPp2H0P86F2dS38xlM4XrB88lE4Sie7ihpTJiElnqkIxSvOAcLvAVn7lk6wP1xUVckefofqcwFAvcKxZ4aFGgYcCNwD0m9ikOxn1hv2K5sCHdhzFJrS0qAuQGpJcZ0SHwLlArRXHSSk33JPiN6a6A+kRRDyDxpWQP75bypYlzCh0gBTmwpRq3WtAxqniF17tqdLD2ovG28owr6Pk0qzAyZLbUY37EERCilzSQJKHP1kgbRPMqrOqwNISipwRcdPVYqqQTffCwxgAUSUJMfjR3STKKrLr/lFukW33fqzk3ViuubqjUG0pFaUyQJa4FxyFr8eIIcJdlz0bg6tWEYN4cT8YKXfH0ygCTVwKYkCXprmxvrgfmJbWBE3ddO//TIn1LPHjKJ0b0lxLTHrctaHvJJM7XcYHbFLsFi2kHjC6a3SrGaKgYfWoaChuXBFn7CX9QjuNg9Vfp3+9dw4OmUCCX+uEzEnyR1fZbQyOatd2HOQyx7kDK1LcGWufEDD6qABrSz8FzA3XrE8kTuTqFp+Vr2dgJQVBC8NKOxvgo8KUzF5kA9gf6MpUPh4KABAlHJg0JP2qUK1GUboShjSMKt6EFCIKvyVitWsNEtWqh8ZUJMxY+LS6FdtKAqddQNqd1WvrBvzWUFEzUv7bqUgFDZSOPyBr8a1W5Agc/bEoyF3/8z8AwdShgYLOYw9kgpmCS+dmwhurOTca4i5JSxeoKPj+PjO3OyngKZ7DipIvVQb2f/xX/6Wxk02rDvKu261ahKq5RO5uts5wVBo2YO6obAgMHilghVFD+pi0GspM6KLiN91v1nxQdhQY/uA5U0KMFz1lqmeunXRtx1QoS29u7E5brWQFYAyrTgNDuUAf9O7J/SuetTct17wO8gGr3giinEehL8ihLiLut6yy44UUUVt64vaHLPzhqIRtVUFjR4m7ANct/f++9g7Oy3HW1j6pym1JDRbtF6CVq0jCwit8plpckbkZwsqY5rYUDpV/AU1w9QCU6dWzLYqUi5rakSaCIR05Y+4n0eHjvlFkZdDjmr/Y1VYhgvuEfqOHsVn7kx9LpFMmDjw2J5/iixan8VOsBOHkAqO6RebOZl6ALtuoWsVBm1RddfGt+t1ewzywauJaokKhwF7GRwBZzqbJNj/+h3/N9vffSqGD6EYKuSeOs79PGzo441rSSh02KCwfTAM1YM0qU565C1MORfquysDbyjNqa4Q+u8dxu+KJD2YC+mw64LTBqYJlhjEEEb5EX2RyUfgWviUpUYT5EhgD1oq67raExnx2ADZ2qBoUYbGvI9pCBBwN7g5WZYO7ih2EpISWCe1mpn4IxAxmsznDyECiKGtmo62y2f5PKnY/wiOkjtsMa8LzmH7WmQnxoN0PYFik/ot6C8BRZZcPKKaI5E8pqRLIBO+6d5RLtP4ymXQAaueiNvsdjLt6v//x327sVxpyNb6Tq1vbeozmmgqXtw7p2dQshfIrHgggHXTppQ5OwZPhiUdbwyZ5+bDq4gqaRI2QQLeMO+D+GdFTQUve/SllMqHT/vY3HzBtUOWXoWeG8UlpWVVITzGDhuCnUkLWUYdC3g/zknL85Xwn4YtyFKbqyKnacTg3SKDgM3Is6VG31dDbKeuuJnX1Xv5y3jDrEY967U14bbG1pffA4x8cX9iJiAOwvE0DtZTBitQ8RQgEajJ63L9Qix8BULn8prjKRihA3pMk/opnXKERUErL6P2SS7i3AVd6BUlD8fDWvB+JxXrd67YVmYG7ravKTkRFjTiYAIZsP/TEzcVeC+09eGtaZ0Mv8swPyjgI3dpuFN/WYekbGAotuUMq1nG6a31guEWOUNogKxk7qslL9QrTSW1w78HCAEpYH3w971RTU1NcVVQBT9N/mvbNn9ZE2c0sqUF7qKFlNS2QDVbHAg0baKQo8RPy9kP5mdDWztNa5YCYOS7EDLqyeeKKQsfUaVEyQpzWGEFHMyIw/4uHV0sYrWxoX/lJlvPAod2a20ooAoU7fWa7ucdteO+gwYNYMSl8VXnppw6/wkT/aYD2HSEWY9H9inJuM5jqG/MrhSI0eBJ/Kkxhx9071Lfvzt6CQr5a2nkqPDCPMMbdH9kDsB0PWrZ/ABS0lXmNRWM0NF99iQl/SzsWCeaZ3P2KEidfnf+G5Sn3+KgdljmfDC7AaJ1P+kAsjd5X9LLZSy3LswRVbA5TapjL3YWAeaEtJ2y6K+yZVlvmZi0jB+Nl4gaJ0nn95v0Hiq5nOhI+j3N9eZXwpb6uXLplC9xiputiN1BGPTWtrZMXEsgVEgnKgfgafvTt/jpdooAsz6qQWWmEwLc+76VLn/V64LAmtz0Y7BhB+584zteGe0vSJC26vqZFtbbv+OFU0MbOE0Fg4ghkke0l/Bp6k2G3sZteWWG0EBwj1mGUA4+nxtZgU7oQQaA2pFweI3QnyrM4z9RDDCToS/Bz4N34EGNrHlwVgDEBfyJDwTdSgN/QGQNol/ix2ZVbLiZYmUYXyE3fWka3DGKiiwN9xJbEOod2FmX47cM1ESNuNvDJEB+DeOGTwwv8eVSCQHpwsptMhjHVV4P+8IjI+mCFHxdPJE0b9yitCkgDt3+Bq+Qn90qbKx6kZB24mYlkgPMDYOMr3KCEXwW/AuLhtQIFl456jqoBF32cxWW+4aVjSdwieJ26Ww6X+GVy6RT7JDelSMJsuuL9ZNQ77deCUQjQKuYLV0tcCe0I1NRF6QskfIa5uDqpwzztU4zoXfQxoh1HL2I8mONZs1Sm8wU+Jg0A+yz4LRjfQRm1q7FX+2AAnQX/7u3gG+mzI7UsolquXOf5WzkzDPlgnh/cgbQnP0nl9WxzjT9G01Q+5xnKrBAW2cCHePjE4G6GsSncFkQ7LHRwvninjqV88+ZtlUBOnygKOR1KEjntS9I4PVaEcHKoNhzBAZMHWkwDCVJJRfj16Hh6PJ318fJXaOyZxm9ByxhaVmKL0+mT6WymW+w1rV/VXLcezE69x9MKvBeyP1f1J/q8f9jfCO0Dyl8th7H90Wx66J3i5Q9//0Kz/+8s9hpmQ0mmp1LYooUBFOAuwD2txx3Q/czqrqPcR0TWAAxbLCfjkXBU0gJGEArqL8X4FR0aW5BRuJj0QGI8H+PC1Fphnh5M9fjo5MijkD99RvYAdFh10PBF3UOj4dZ3LSvtUUkuGmnYNP0MZ23oFH+QZr8DOYgxlJNTEmbW6jXo7tfwEHOObkhSdkm2SR2+VZeo0M1NPXQD0z/tjy4aJjYtRBldeBZHwS0mJ7MYBpal484KxnrXwZhTMO6EYNdi3mESXQKrrTAJb8tIZoZs2aMMRWg/tI/r6B3Yj+86B88rfjRRRGU8acxDzdox93BraHQY37DTIrexyGEc1nMYNyaGrzpqrQ8PO/XhdB4N+dA7fNy5Uym3MEOdQTmATtMoAN9mM2B3E7jO8xXSxh17Nn2+IjokBbXK7zAX9tkBzlJjAaUIumcgZmSqgpYshWUKnVCj56xfklWNEMksDfgdGWxRoA9k/EmRu45YGuQGxOqmaw1VBvFdW0Iz9KFC+ClhssBRR8lieQgOcYrGLIHFdoSLdUyDsnKI1tGeeEZXN4Nxp99hN0Og1t8Bdd/C/QrZsG8P8HZYu9WMoOZyfKwf9TyeLniS8Ntx54gdIYlTj2i6YSC9JXFaYuP5j3/zt+w9Zfh7UfMkEC2xypumDGpAnRSX35UeC21I1lZW9SBFPfTw17+XCWvyFI9Gyd7mlPlqYvCwPxwOp+tMUV9QOarm0jeTf1tITFPhCQCmB5ivS6m6rEh1f9rINd+WU1zL7W9m7cPEXdGbiuwaE5spef864fEIf1DWPiXKr3DR7tY6acGwtjlQJ4Pjp6wbygiVT5quIIiAgybgzU4gJpbG1/L2d2rZqkZMdXBXc9pt/DK6AXpmfQbEzFavgWTxVLJIrsQZ5We+Q7elFOsb0smf4OGFZh8yORtRxPqAJM1Ld6tOR1IraRZkRhjM7muk0N5YDehAgyUZ1r1rnBzY23vxHpOL0dqcRt6txWyZdk+LzfHYFFvVEtyfDE4GHvtn0kPk4FbvtXwBLklP4eO4D1iotH/KamR4t/do4SVsMWCrMsV8iBTMFKb6qknMVvXzHLrBEFHJsFnqe6Ix3OnAGx49ZWt8p7hLvymYCXHINDfh6RMEi4kxfwq4OFx7GkSYaKV6kbf1zqTsqOFQQVcIkX3WFuXRHPfhKlCQyHjSo7VCL+q074m5pQxj69ET98ibzbptCzWbzaqP97SxVBwIeNJY4HX51Hh90wMp70XXI2Sx4RG8V+M4PoaRJ/48BZfIRgJmK0XYd0w/D/gUfq7qnQzrRHa3N4uiDClRMwmyaq0BUrzaU8z9wHNyXztwG5yac7VVFquEG7GkKFdi02Vqf8lTYcaJj+FEZ5GLseGHnj/HNMdQ5FkCjjA9TQOwKrdHHxC340kJ+lUUzQPxNTw1ZXET9DmMQrCjpkt5mPbQuMfsejsF3V50gHztlJTgkElqKE6Gb9veOx5PLiuNSmBIni3QJI0itLYGGpxu1QRHa+kQvWErIucSVuVtAUg1qU8SgzIFFClf9PzKVxqEft8YC5FXHZKUN8V4Mj8LRHtflb2COM+2Ib21wWas0wkAJxUgYz2kr5YFcIezQxpks+32kd4HusD4DvA3otZdCPcS+F5ht2X0esm3t9y4/I3P2glhY6PGuCW/TwUlzVQqxaCFdj6hkIEznxrkEx+eKNNRFp2RJ2b8bIGC4Uskl9QkehkbZ+eM8iTwXAVu78OAiL3H9BNGl6ZjUMatguFjOjbM7vj5yotc8EvDzEaitt2Ap+k3fgr+i+eZHZwF1lboYrzHE8vKQWE9pHfRtSkCsXR8b2yAVjUa+77Q5C1AAdmM3vUj1hz0d+Fa3P68ti1f6+h7jP6po7KUyN3o7eHnwvC4GR6y/vR97ovEi2rgdN8v6PyombpgTo8PLQwFONKVPhz2LVbMH/VvEwFVOB/4tBZF3NDGNNSpTWND22qiPXzyBuv1mESDY5noUWQCIE3cxkAssR8Y1s6bs4xKANGm9Nh49+rLM6M4oTw87Hc3wtGTOHNdkLF+FJrGWzyXLs8kMFOArRx4HKuiJCrnBo8t4EEM3PtsXdNNfSBB3NOUuA4YvgVNL+VI6vjZMu3BcMu0i/wQMWtbknOaLNKY/cv2R1n0mzsEJfcLdihPq6x39rp27qTR34OAO7q2FHTxVRLFJDTuRcJEHiGx6GABnTEgtKANI1/Iswb0BtvswgbqS/ymhFN0gI/opgnfuNghdZ4HaLRtGrNC6reAiVBVXEAOMR7Ec8UBb2a+8+cFdzXydjZQlMT/O/TATDrMgslbuHsuB34myyLQ4UA52LUTNff+KwSmtMCN7bShjgTIcb1QKteUW2UFN+sJe4VyfdCIKhkZY1DQqiAJlnvA7NfYR0kVfaQbeOh9/uM0D/j2o4Nb6ct8/FhViKI9HfEJuxsedh+2zmcB5RQnggcPWGPJkxKd78HFFInZt4cWO7bYwH4ijwGO+3Z/UKB3KbIkSh82trd4rBpnpavaPEDEV462bFl0VQ2Hpk/ZcBuq39zTSwMRMOuhDajo2ycFKkpMoLMUgprn3sOw8QKGiB89AAl0kKXKjJMNRxjl+Zb3r5hJNWD0CZpdhFGbZGrvRM3/LQ0KzQuMWT6UxyR76eHikfnifAGfgvklLRd6z87e3LN8FEDL1qSVqgVZCCy0uovhnwOtUNW6B0sHLRmUdBnJ8pB0LEIe/TweDEE4DI77rN/vy4pQgEDwacC6xpzNtYz+XUUFdURHpZcAMqHqkcx8XOnnaYXmsRaDx9VZ0Z/WIc2Jvpd1rWwmC02O2LEqlmnp0QRU4Wj7OsHAGip9YmBmAFIuVrPE31iuEvPz5ZJhWopaxgku38Xk8KL70AVTK67PXZjxTfeegT7QsCzOKW6TUf8892FBMP7i3YcnhSvMDNkGUU9IVnN5mBT6rawVZ0YxHuzggSWrRtEqPsgGr5z5a4xV1SpSg9XVZ1TTt+dHP0ktlwz4a0xocEH84OmsUUV+YEGsRG4+sVuqqStYFnlIozxltfpye/eebGzQKxWuLQygsnitmqSqS/f5H0LKvCAI9/GEPgm6baVptRhmJzJMrNwMcZopPSnrOhTuY3vRH6PisM5FiLac/wnmFcy3dEE8vD1DsVpplvITZf5udzI6udgIF+v53GynNTxPIQvTpibWHqBAVzqekAV9UZqAHwW02Wil1Nzrt34qoxBRus3FnjtrmD1zMQHJ/0TrL/GWLrfgbc4DCeAV6uTkVjtNHyIXi79Esrwdx9o3QaQKvunM1gdwiixwlY6Hhcd1jIEJqv4jM6zFtWLsjUAXmI8mB/s1uOZ0PsRxozzMwADE+qGwAiXR/8s355XzCLsg/ewKKyigGBzt/QIimM4v12y3Q0rOsFhddWB2G5lxg5+oQc6pPLT547/5z4M+HpGlqqD38Lc8kV4b3QCzjR4frY1voMd3OCy9P/p8Sx900r0OXtd9lr/rfQx1H1gd+iFoUON5QdYM6Ao0A3bxA3ZZP1mBAhr/Ol9OcQbDw6MCA+/F0tfHh4VLFY3G94QQiioXWzXSWSB4hoVltjGtKhhQRzAW1wYnCQtPKLfguID6NTQhA1/WtsHTOFvgy1ICNei9wQnApR8aPmalaTXqozRPfbb0UVpkCWpt84e/7/4yS1HUMmjaaUc4nicoSQZ9Gt/RUb+w0jBwrT98sK9Rp6935SnrndSnqqhQw+CRqqUOw1KvDU31JVkpN4zJ2sEPM56+ibDWSvU434NMpqmjT2/eI2DpMcbQQMcZFK8zSi33deUY4eaedo7iNMMn90L+WebRvdDX1O05pX1zrImaiXs/l0cnqzsO933xUI23janXR/9bkbA8JZ8W004OmMwh3cFsSMqRbZjLT9jpEKkrYGX8HfY4nlQ3NTCNoE1pY739PG2MckPJyv19ZZPu79sM8IKVfhN/KRIqarCjnyrLK8CC5GJeLYaA59MDWd2iGlFQf5dgVzsKN3WU8fSkRdXU93BwQwwM+HSHvZxBUfIB4+9lkYjN4mLpYB0LidjXH95+Y64VPjv7Pv/8R8bBeBSuAG1QKUS2gyU2rNY4kGMqImbbRlV48jsOqrV6xE4jbKvbpkZanPvaPFIc2YaBtnQOnLYmmhXLfwUswxOsX20C98tZYMLeF0U9BJaC4JSqYCcqaxUxm4mtxXfX9QLJ1+Wg39gPf1d3s3/4x50U044GwhX9gQ+JGxqDuS7Xy83Bky1G2pUDbmwLoFfaw90Opi3GrM4UYLnE89+AgJVJ9fcFwpc6rkzkUT9yURF8y2iTxoI321xDfYqjXdZ/jKbKcEEr0DQMVV5LbnXMsORocOvI6KZMZG8U7IEJgpVgMVQQ2s7HdnUvWx/x0q0xqJZVupId2O6Ch3NhVju1mP49aS+wBZzcrCqE+orODyCzWSQaLSXHLCk5LEVJlqQDq+Q7S2kSOixZTsIEDMsSlWZZ3LCVyyZ/ooqHpav1i5Y9rNvcv3Ttw+qWmap92G6cT4AQyzWSK6YXAmd3gac8QjdPwFh2b8tw0V5r/KFsSCeP0DvAUlEiITN8f59KoUrwDRlsp7mLZ/vMasEii9HoJop0qqN7yLjYA3omOlb9ljT9czore1MCQ5FzVeJYqtYgPKscnq00DvhyimdLzUqNCKs8QNzVACZzHD0FcSrMCQ9teWbaLMsM6E/KWZYfTCtdU20UgDzVfEqehaXdgAt6BZ9b0vbesDCVs7oV2MVpZD0YMnmrh+iWkTwU/9MRQDDAs8mFCYKVg7p17iPUJb9xSGbrhAUlRhHU3s8vPE1FRJM95/3Ld799+a5ehHkGIsjBbFITf5Rn/174+EcdsBAwFV/GjU5XBB4nYGgdZVShaZqnFFIEg5s20gJ/mqwVDU4j91KUJSsvZYENP8RcXtyHuRbTqmGFoRQ9VqpcedlSK8PDck946L9e8qm1OgwBBQqIUtHwZzafDF0ra5A1R2VkeRgK3AIvB4cnzNsHltmqYHCmx4CoiBEVCelCxJTFCF9fsKN+t5H0JnFoy1/SE0h3mHZqT32sHWL0bfoPQ1st1qguCdyGnG/fU2Ghddi14ioaAoxfkVbA89BdOHKZaXrjJyeP9cRoouM1+is4CN6VCc0lX0mgJp1hSKiA4bgyNfWY+pL4lL2rE854JkdsShzBeLUIM0eWIHQw57H4LLp2BOJAPdDpRYO2sCZFv4Xn0PmH8aRZQaG6X6EEhqaqiWYJKo+KrL8uCohn3eKPNHpFsveNqP2dvUoTY0vJEvisWp61fuIdRNRcUEZT73v6+S1u4txsSboxFlkWp6ODgzmQbD613WhZGUl1UAdgVQieivQA08jT7ADdeBS8B5u9cqPyfQ8Qld/0+NJ7fIQSlHJiG/VlYK7uAmQ3ztJi/ejJ8XFNusLjOqmCHMK/3JiCZAYtApPNgaDGBsezrZp+B/VaXLp5vQ4KPlkrKZwnQcpnwjzt/r9kgCrNM55ni7GJs7KKcbZscWMzZwlkgBVV2xJ+d+eVn8UeUjGS8yo9muHxcmo0mQa1R41jANFwS1XK1+Si5JM6B1Wrmbn1wg/nVBtnQgRUynuj1wujHv6lHlkfRj6Cbndxx2fEJMAjg+ETWrfBaIVLdbdLNFsWZhhXR/jm/CXaqOBjJNXn7z98BUitFHio/NUBrH9RVrYr9Qz9hYR6zbuMPWOP+416ZPLMJPakKkXYSAZBvUQKGaBYBlGAKYiBCzMpxMOE9z71e6e9iy++s2F5ytX4DkUGBj+xC/xTNMZauajluiKSC760McM+NluceLBJeFkTd1ORWW3yTahUOqpnB9VzZZ4XW2rQ1olS0cqFwtMu5QxkCV2j/Au8jaIFKHwLKwOnvFbvUkH4Tar/BIYIsSQYk38DOKr8fQf6C8GJzIuObNs2GiZw01ZqF0hrQolkjRQE7aJms+RoJf2H69KSHGhgNCKUvC1LtCY3lFqA1lLylwPf+79M6QcK", "estudio.html": "eNq1W91y20h2vtdTwPTMELBB8EeiLJGCvLassb0lr72WZydVKpXdBJokLADNBUCKMs2qfYLcpCrJXWovkqpcpCpVudi7XKzfZJ8gj5DvdDd++CONZyaZKg+B7tOnT5//cxo6uucLL7uZcGOcReHxzhH9GCGLR26Np7XjozFn/vFRxDNmeGOWpDxza9Ns2DioAVgOxyzibm0W8OuJSLKa4Yk44zHArgM/G7s+nwUeb8gXO4iDLGBhI/VYyN024ciCLOTHp2k29QNh7D47aqqRnaM0u6HfXiJEtmg0BqPe/Xanvds+6DcaExbzEO+Dtt/Zw3sYxLx3v8M6/u4+XrN51rvPD/gh53iLpng7ZIc+Y3hjnte73x10B8MW3sQVsAwP/f3BcufBYiDmjTT4FMSj3kAkPk8aGFkST+yB8G8WEUtGQdxr9cc8GI2zXrvV+rY/YN7VKBHT2O/NWGISpVbfE6FI9DuosfpDsKXX3pvMjfQmzXjUmAZ2g00mIW+oAfucjwQ3fnhppyxOGylPgmFfzHgyDMV1bxz4Po+XO/czMVlMRAo+irjHBqkIpxnvYxTETOb9kA8z9ZRoCvHoB+kkZDe9Ycjn/RGb9A4wyMJgFDcC7Jz2PAiMJ/1PjSD2+bzX7RNk4zoBKP1vueOkfLRYQbNxaikSq6/41mvTQUUY+IaaJQHlk42E+cE0VbRNmO8Tv3cnirTOZK62MwbTLBOx7QyyeFHZLUvAnglLQPIKlyHlYvfW2k6PKhvh2WhjFyWRIB6D0cA0TVKgmohAcuJ6DMY0sI3He7EoeaCJcsQKSfc7Xme4yzaEvrKmB/axQcj9hQDaILvpObvdfF+fD9k0zGjBZEH87bXx7PulqBMesiyYcTl8vM6UrxfBFhp933AiHk8LAcci5v1NJVMa1VLK1t6m+b9SB/bxHAWxcha99iHNRWze0Na235qNS4tg00xI4h0x4fHaCQah8K4wS4MGs9WvksMqTF/vRYac8XnWkGbRIzPatGEJ4HNPJExyRvFJE39AekUUrx6QzlThk1xSaOkdKlgQ3xvTkVeOoIa2aGC+KI1YGC42rAN8Y1lPihGehM8nt+iRuCrEVMVBw0Rx4zqXRwtoBgIkRVt8kproSVuTbqlLPCbrHYokUnYMneZ/ZzYwY1W8z4qjka7IDxLuSfQgaBrF27xX7thIY5RUD/dn16CQxUGULjacIElm1c/1P07TLBjeNHQM05hzFLkC/Wq7q7iqXE06a3bwky5qjShnG10IdVsFKcfVwP3hcAhUQTwUW0SoogmClrT5faJxU6lIIxA0uZJ0LsW9Pp07N92208UuIR/x2F/c5lnkeq01j9Zsf6+7bvtVq9KJgOf9YtdTWO/acVZcYnG24iyGH8y2a9YWBdXZA9AarWWOIFjkHgiLirRiw43sVmgJYsla7ePuw6i38TSIkazBx6xQt4Wq7Tq/Kef88LvLHY/FM5auO9vfRNwPmGGW9re/h4NYix1DZi05Z0iVDnKfcFBkKgcU+I2NyF+1icNV+ezKFUp3CanSCqU/2zeounh9zgq+donPGCQr0ZDGtaPTG+y2FHQmRJguVn1OoaiNuQ5TxqoHyQ/V3aJ1+lRpxkZ8wPwR3zhbu9ta41puWSsA6ydb7hw1dU59lHpJMMkMyvzdWhBR5h6xSe14oV/SWm9Ry8YJ57VebZxlk7TXbHp+7HxMfR4Gs8SJedaMJ1FTAv2m5bT3W06rOZgGoa/GnEj405BjRc1WqJo4tYjT5s9DyecsQpacNj+mUbO2XOIQknicoqmqE0rN8QZTNALfrYFdKCwMQw54IUtTtwatqslJqlNSOQ0AJQ3DZxlrzLCQz2v5AhHXjt9BWaYJO2oquON1eCShYNTxm4B/YmkBtRW1ipgEzFOR3ooRm9/Ujp8kXhCGt+87DgATZqy65VETp904NRixdcL3FTdCoqpWbKBnYXa147/985+Ms+kn42//+N8lIRUUlGds42NIhaMs5mp5Vbf9IAQ4hwUGIgGkfjJMeC/rDl5iFX7gGT2OZU/yx1u38BMWIVnzWO34GR6//Jmeb4WGvceAfEM/hpmgdE1ZYmRKD0qyFE/v5KzHomg7Y//nX/7+X42TL39GNGA/m7ueW2vZrdrx91T+8O3nAEz7QEKdooAJfXYr2KGEOmO+MIJPf3TuYDuAGxVoBCbnVqy7Xbt9UDvebe7dCtKyD1qk50kwuJ26IYqv2vEJq1rgTzMeWe12vp/OybFBnHfyPMehBLCyX/6T+xmKu5K+EYvBE7g7Hoq//enf1uFKT14zpP8FsXcXWuTwu2XgWsmruit5j+5jlLncZg65Jc9cjTZ92fUYQ67XvZbRMtqUC2nMBweFBynPLWPN5jjFzc1RFZJKh0xjMhbe4poKTy0D65oN0HgishUv/QSRsYFBtukEpCFm8BMnsJakArCB9Br1BexehF/+YyTiLR5aCvKKh2RUf5zykGdiO8wY5MHG/+EvxgmbrISP0kvnPytRWIVLHFhFYOOBwVLj3Yu3p6fGMBGRUZdhsd7P5xevk0GQnUCUCRi1rALlgdbTk80VUATcCpbnZ+++P4Mmo6LYhiKUU2mzBFtd/uztk5PXX7G+AreK4K0Q0Wk8CxIRw+ayrUh4OZ821xYobDs4apoZ37ipe+wLbypnIKfk5hyy8jKRmKnV11Dfvzw7PXcXcOu9i7oy2/fXfOCMwkHdzgdIR2ESq4MqEgSf8olLW+YAQDORSYAe1NGehinc61EK7hhiKrrrwXFANKBaZ+8ZQroaXeaU/u7lyam7WKexV39+9tRA7OaJMDBomAhtnjAeGr99c/rc2n4GtUiPGCaW2cZz4YvMNt7JdO1jWlkJx+MMB3Os+v7p31FL1+C52hvmD3GQ3djGD3HCWWgbrwJkaAKLdwy93BmE8BBY/JR+OaK6GrFu56QiLw0qG5XASCgohgtHDD4C8vXT3wIqk7FcVLZ9D51KeApXuus7aRYC9PzdmaFHv/xXbOw+Q3kSVcgIhd9+321pEs5eP8vZ2m19uwLVed/uVqCi6U0O2e5+K2moaECvrnJCAypujKfQAeAqVaFXf8UmDEHUkGOYq+oEiIY3N6hHwoiEVd2gtRgwaMDAsRAwPHk0k3iXcB+vCR1wFvhcvEcaEsQscaLJHpb+gcaUDshFBdgI1rSLbLsKhve//qeEAPffTygZYs4kHgHgZYSIFtMB5KShJuvLwg4TKXaeuDG/Vj7M+ZEPnp+91ePmgsV0ERCwtJclU26ThHgy41Dka0Svp9PhEFU8TS1htjk6FIfZm2DOw7fUfzNfsWzs4HymumMoZ+yOZfWri84R68wgjnnyo76HwOMLaaVV9GKaTabwkwim59T6dRXp52+fPy0HS8Rw6hySnIBgDfnk5PT8+yCMAu9dOVehRAZZDDs8lq1gl07Y3yk8FlUyDlZhwckYlZRZLPXh9EJOQIUbS1HB8gqHz+ndtPpy3CmzhAqIPIXZmqt7lALTJEp4VAF78+rt6avnwJIw8pw5EdQzInC4Y5RGrlzlkLtWO9P6NedsWjZKuD3LUebO8w2hltXtECAm1N6b8RMWYVMTKWQprWZFWoSubXdarYJ2LwslqpUQZ2IDeyvvAK6Z/wxlJUlOioCGqcR+izwi49vGzicc8mo7nXzjURJUWfscry94OIFyt1v2HnLleYfvttVDp7PX8XLJIKiZtLg4wjAUomoqr3g6NivsQTXCn3MR8Sy5MTstHN+yK2LXSkUFFAvN6u3C0pI9X5E4lB9RjunM3YY0mzcvm518DgUUWK/wqKOXhEqQgtIxj4IKoS/wmk7GYPIZCQeKNZT/4ci7+G9vz3b2q6em5aX6TuMKqmd5g5eFa7g6wDCNHY+l2QqJGFP25ERsQgZOhg727B3Y9L/qvoAttk2C6O5tDw4Yw7btKgIsAoIdRCXDm0JU0zC0I7gb/cgkitRdLBHkhXdVtTh6hxkgx66MPhXzXQy+gD4d2JRWKs1R2CiPJlRsxgKl2+c4WHEAlVHJ8TIpM3Pb9CkTkJOVjAuzcpw49Ix7CGrJGyiBWf+lbZgwGKRNibI5CrNhs271FVlyh8rGEoZYN5zGkkmGbDmYV9QVNBTJZ+5Cdwx6F6XU7YuOvWfvXtqlRKBNdttp2ST/SztvHuhFHd+3dx0s69pQAFo2GPoHWOZ0beeQlqpllPoXHQS1lrX2YVSAuGjsYWO5eL/1qEV7AiMMSf5T64uGQoXYXexxsWt37Uab1g6H3e6jRzbJ1gHR7a49ZGGKtbK9UFnn7NkXLSzrqlXKdLBuX+6rFi0vri77IFraALlvYvELPjfPLlqXyjToPiBGDXnjnl20L+VQXlNKm3Ac5+yic2k50TTMgkl4c+6xkCXmCwTJIFrHunuphqtY9yQJNLqCuNF58MKmf43dB0BG5r2yrHupbaiSwGP40eVjFUJ6pO+bIC8rKPblzsHQvEfrrMWWPR46+9TcLDRsAFszrQUM1ZlOfPhFOLwkmP8oEgRUEiJdBMzhNa74aTTJbgrbGWxYKO1NeFA8zhCkuCnc4wWIEU6Qkpe28pfzKwpSvhqDnV3JDDbWBACRAJsjJBf8KYVkSnGIyP4Aw5MboBiUwxZlAOGNonoPc1FJv7WUpSuHZmCbkQ4KP4G7APvqTYg/0xjMNAfWckm5FKI3OCvmFT4PYQncZJ9c2AYP3XZH2rTivuao5xKmEc9OZLO9EtT+IIsysNiy0xxIZmnbQPovXJXrsbmZOjc2cgAlGuXyqpNzmwBS55P1oO0cNuVMxmJKCJyhmD3Igx+yWwsRsEDCXPapmKT2GXd5WB2QoGzNADxn/lDCIPk2maXgPZGa3Hrg255zU87qkU8PC5h1eHUkJB3wTuCGkh5d1GHXmLPE9ZtIfOTrkN4edNWL0rE3ifioQpmSqalynUIBqxZC5VQKE6F4lrkte+a2+nep+Xff3RPOFAn6M3hOh3olMC+VBLmlevVnD92RwzLsPoAypiWrPOhd1s9oWjaxHutfNdHc7d2xrLlLCljI6SuUBbDfmHV5jVO3HJU8vnt15n74ZiH5LZNiM0NSKs4EfYh0jq3jkWktDTx8+XM8moYiNf76F+ObxWwL0OzLvycIADw9GiTH3yygcwD6HtmAb3Yw/eWfDBq82Tb4aWUw+vCQrNkkSh1ZJqaPP6h9K0NLVUGmH3r1OsyfpTexZxSi5HNUYKk5tBaQgBZK4rJrFiCv5Jk3Nof2AuIZC79Xf3H65Fm9NGcUPFdLjxEQBKoHZdRZLnfWN0rH4tqk25NK5PZY7KeubKdc0NRlnzRqGIRcZjJ91LCmTnENMVTgFvRKZjbOmBHd1kLCD/uDhLOrpXb2NGYpinJ5UoIBecoOqqMbqG6dbjXrpXqM9NF1NkI/T+gcJiGETUu15pBCxkKrgpXqkxP94dqHzZausaI67VbrAZe4ud/MkS2//aD0FBvAlqyFCmsokcSMy5ElZspcbw2gMtFfSwiXKgjBeOSK/t0BSWxkymJbhi9QuE2RdUUn2AGFjRR7XwYXoCNhuq5bp5ZV/fPn/HUc1C0ZKWSpsVaxnMPN+sirykpEdaWRiB0OPG/fBvNG45inac/pdm36djBUb62u4tzK1rqHVqcTffWO4AkkKatc3ViobPpoiVimdiozexJMfz1sRbH0MlEQOzeS3YVLumm4UazrpnIQ7pMKutURIC1LhH5ZIvRJ7WQHfMU71etaeeALqUEjwZ2Qx6MMUtWYiuM/yUFe0URxitwMZMvfvai/5SCJmk8nqg9ETyJBTYyHcxZOfaafMvnwFGYpH16xWPalXqBQlgNvRMrfv6tfqk1WSExFkpkmsweWe2zKjZVzfw07d+jjUOvh4aH17eFhY212sDKr6S9dhkcuo7qTtdBMvPDkyktXcsXxwmDyRM5QtJRYcj4M3KK14sG9ZFz3Acy66snXKUuq2r7CDIOBd/G4WX+PsxsSSsTYB6WdiWOS6zEVqLUizmrrZqBzNQmcU55L5PJx/tSrHhEpvcaqjqajNzU9JDZyr3Cb53AQ2mJvcwbVhDRfodtpRDC5GNCbf5Lo3tMw1Ul5x3KG8JL7qLpkGJkPz35EEmASMGUDK8B06cCCOFXgRThWt0YbDnzd3EtnIzvr9cd1+XVJvVenu7L6BrbSgNYxPa4fDYrr9mOK1SfMZ4Z0SoZrTGMVVx2aeSagkziWEfFPOElsGyEzZOeVBn0BK3DqPSnOdfJoE9XsRayAn+bJhG6GjusPJcSZJNS0esq+Vc6MallWwRDemepB6oysf3ugK46//XrjSRhiqfy6wch1GwlvcsoQ3Qfu8aAioUyMRqESpz1w6LoVAnVmOJWM76u54sopdIKRcq56N6/YhJzmL0odI/dJkrAbwMlfs3Tz1uPyGRbRK9+0i4y++y6SdgLsRItMJdSIZckBSs7VgF2/X3+YF7kjWeTmqZylo4HOfC5QJ9PiS2oomeZFbHuXMPYPR/IOOMjvbivXr98svCVdYwZIA+OlutX7YDkfBfJ9mazphhF5axUGKnwltxDnzED5oT1EfNmnBIjl2Y9KJwjDd9/ph3uuyyz9jFLA56+nKPJ3rT5zqI2emZYcfRnToCM3kkKSZLCf0qHqh0pfq0NVF+q68Yb/BKsr1Yf2HiKW5fM9Cl76sIoZ16l7cXmLVq2q0jWqhWk6NoUFSeIlJ/baPb521DfiufO6tm6t5u/lyvr58/oGZQoqFW+j1De/Rod7F+XzZcnRyD2O5C7SKbhgh07Ml8V2hX5cr7eHy7LLXp15ytLA28zBDgYHg+HQLrZT2VHlW3o1UGke2z6fZOMfkyDjPVUUWEV0vV5lk0oprx3VcX8tk48OUkxKsK7JypZF71T6O7euG371yt3HpirmH7qva2Eeiimd5/CTYoLid8JGMoaWnVB3oFVAB309Llx/e7RC6Fbh7W6SKtTM3eP5tjApMVnSjoVV3Y0YoqfJ9RQbYfx0hgcComsXsy5PWLcp2/g/Ise6k9cX6hOoy9t4TYQsCvmVUSNcD2XLr9nH+4l9Vm9jVKZDoTER2Z2JSUBZZU6aR2GaPiJCDfGzu1IaNHM3Jz1nbqsG4hz1wIsHTusRNXbyBHy9R5QBPHNu8O/TQwB3u6oxU23yZGu9mtINKCoumI1st3IyB0lBgCzWBjcoWP1uGg2octQtOZv/lBjuzBSqspBlfyVJIDUqJXGX1O6tvm+VXzWKrIJbS7WPlyVr++Qp1E4l+1ylQ5l57N77ivy0vxWmShf55UrI0nTpHPmOjTey6C0bb8Csb7xTxIFqU6C8+9N9Zn3xKMvAErAIOo505e/gdLUtbYOhb8rVn+x1qldnlb6E0u87uw+/Mh4uIqcSklRYiRwdlFx5oymjie6Af20Hpf//TXROYbu/egAdN4nmXG/Gm4ZTXFSrB1Mey4Yjyb0Qu7WMZXXK+sYI6u6W626oE8XoH96emfWAvtlo0vcbtMIX1zHVGW7dU1+pyS87MCHJMknLN2MSfUXziaugtCA/x+S1vbv9qv7u3vTXfp2x/vlH0fg4EwJJeml2Pop3qhTJrz/jYcZMXUSjVqjW0v6ms72N+7T3jin7oHIjqB01WeX9rGozmtVGaz0JRu9l4/ZjKktf+QteFi3W0qOjtCxudl8PiDnOjIWoR1UzmCXeOJiJ9PPnhc661lqpG9+M2b/uQza7+vlS9Vul1W+TLi3sqr4L2EaVPskVv0lN+nTNWgNX3II2REEK94JARDUXflXJpTvOyjwVbNHetlTPWOHSPPml8a3seqjWdSW8XTopsmoTNKz0qCsXEJFLZqy/EK5/HRc2Wt4/w6yHVWull7LjQcgvhpcPPxhH8g/+UIUO88zAwaEnsBCL/l5CTn6Ab6r2p5j0o2u0SzKJ3nsa+a+jtVoXDrfur7D/XhoD/NQ5J9V/wxDjUxPmzHQaRaOS11RGzNzfk5Wb9Vnd+vy5ntENAJGsLyJuk+HMrUpGAkMxgtg3r9xjtfbqtrXYRvbCdUDW+6u8gdLMdt3amkiohF9np/rrqSwnn2dsQtyjK4ksZ/RA6lf5zfhGi5JnGFhrDqke2bIwMXVZQ40cizKl4k9mjpryb2WOmuqv/f8XT9EttQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()